# LeetCode 3798 — Largest Even Number bằng Python

Link đề: https://leetcode.com/problems/largest-even-number/ · Độ khó: **Easy** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def largestEven(self, s: str) -> str:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm (`largestEven`) đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` chỉ gồm ký tự `'1'` và `'2'`; được **xóa** một số ký tự tùy ý (giữ nguyên thứ tự các ký tự còn lại, tức chọn một **dãy con**); trả về chuỗi biểu diễn **số chẵn lớn nhất** có thể tạo, hoặc `""` nếu không có.
2. Nhớ quy tắc: một số chẵn khi và chỉ khi **chữ số cuối cùng** chẵn; ở đây chỉ có `'2'` là chữ số chẵn.
3. Biết so sánh hai số dạng chuỗi (không có `0` đầu): **dài hơn thì lớn hơn**, cùng độ dài thì so từ trái sang phải; từ đó suy ra thuật toán tham lam: giữ **nhiều ký tự nhất** có thể.
4. Viết 4 cách: vét cạn mọi dãy con (chỉ để kiểm chứng với `n` nhỏ), cắt dần ký tự `'1'` ở cuối, `rfind('2')` rồi lát cắt, và `rstrip('1')`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: xóa mọi số `1`, chỉ cắt đến lần xuất hiện **đầu tiên** của `'2'`, kiểm tra chẵn lẻ bằng cả số rồi bỏ cuộc, và chỉ xóa một ký tự cuối.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các ký tự `'1'` và `'2'`. Bạn có thể **xóa bất kỳ số ký tự nào** (có thể không xóa) mà **không đổi thứ tự** các ký tự còn lại. Hãy trả về chuỗi kết quả **lớn nhất** biểu diễn một **số nguyên chẵn**. Nếu không thể tạo số chẵn nào, trả về chuỗi rỗng `""`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"1112"` | `"1112"` | Chuỗi gốc đã là số chẵn lớn nhất, không cần xóa |
| 2 | `"221"` | `"22"` | Xóa `'1'` ở cuối được `22`, chẵn và lớn nhất |
| 3 | `"1"` | `""` | Không có số chẵn nào tạo được |

Constraints: `1 <= s.length <= 100`, `s` chỉ gồm `'1'` và `'2'`.

In [1]:
examples = [("1112", "1112"), ("221", "22"), ("1", "")]
for s, exp in examples:
    print(f"s={s!r:7} -> {exp!r:7}  (số chẵn? {exp != '' and int(exp) % 2 == 0})")

s='1112'  -> '1112'   (số chẵn? True)
s='221'   -> '22'     (số chẵn? True)
s='1'     -> ''       (số chẵn? False)


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và việc "xóa ký tự"

Xóa một số ký tự mà giữ nguyên thứ tự các ký tự còn lại cho ra một **dãy con** của `s`. Ví dụ `"221"` có dãy con `"22"`, `"21"`, `"2"`, `"1"`, `"221"`, ... Chuỗi có `n` ký tự có tối đa `2^n` dãy con (mỗi ký tự chọn "giữ/xóa").

In [2]:
from itertools import combinations
s = "221"
subs = set()
for r in range(len(s) + 1):
    for idx in combinations(range(len(s)), r):
        subs.add("".join(s[i] for i in idx))
print(sorted(subs, key=lambda t: (len(t), t)))

['', '1', '2', '21', '22', '221']


### 3.2. Số chẵn và chữ số cuối

Một số nguyên chẵn khi và chỉ khi chữ số hàng đơn vị là `0, 2, 4, 6, 8` (vì `10` chẵn nên chỉ chữ số cuối quyết định). Với bảng chữ `{1, 2}`: số chẵn khi và chỉ khi chuỗi **kết thúc bằng `'2'`**.

In [3]:
for t in ["12", "21", "2", "1", "112", "221"]:
    print(f"{t:>4}: chữ số cuối {t[-1]} -> {'chẵn' if t[-1] == '2' else 'lẻ'} | int % 2 = {int(t) % 2}")

  12: chữ số cuối 2 -> chẵn | int % 2 = 0
  21: chữ số cuối 1 -> lẻ | int % 2 = 1
   2: chữ số cuối 2 -> chẵn | int % 2 = 0
   1: chữ số cuối 1 -> lẻ | int % 2 = 1
 112: chữ số cuối 2 -> chẵn | int % 2 = 0
 221: chữ số cuối 1 -> lẻ | int % 2 = 1


### 3.3. So sánh hai số viết dưới dạng chuỗi

Khi **không có số `0` ở đầu**, số có **nhiều chữ số hơn thì lớn hơn**. Chỉ khi cùng độ dài mới so sánh từ trái sang phải. Trong Python, `'9' > '10'` là `True` (so sánh từ điển) nhưng `9 < 10` — nên nếu muốn so sánh số dạng chuỗi hãy so `(len(a), a)`.

In [4]:
print('9' > '10', 9 > 10)
print((len('9'), '9') > (len('10'), '10'))          # khóa (độ dài, chuỗi): đúng ý nghĩa số
print(max(["221", "22", "1112"], key=lambda t: (len(t), t)))

True False
False
1112


### 3.4. `rfind`, `rstrip`, lát cắt `s[:k]`

- `s.rfind('2')`: chỉ số của lần xuất hiện **cuối cùng** của `'2'`, hoặc `-1` nếu không có.
- `s.rstrip('1')`: bỏ mọi ký tự `'1'` ở **cuối** chuỗi.
- `s[:k]`: `k` ký tự đầu (`s[:0] == ''`). Nên `s[:s.rfind('2') + 1]` cho chuỗi rỗng khi `rfind` trả `-1`.

In [5]:
s = "221"
print(s.rfind('2'), s[:s.rfind('2') + 1])
print("1112".rfind('2'), "1".rfind('2'), "1"[: "1".rfind('2') + 1] == "")
print("22111".rstrip('1'), "111".rstrip('1') == "")

1 22
3 -1 True
22 True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hai ràng buộc cho đáp án: (1) kết thúc bằng `'2'` (để chẵn); (2) càng **dài** càng tốt (vì số dài hơn thì lớn hơn). Để dài nhất mà vẫn kết thúc bằng `'2'`:

- Ta **không bao giờ cần xóa** ký tự ở **phía trước** `'2'` cuối cùng, vì giữ chúng làm số dài hơn mà vẫn kết thúc đúng chữ số.
- Chỉ những ký tự `'1'` **nằm sau `'2'` cuối cùng** mới cản trở (chúng sẽ làm chữ số cuối là `1`), nên xóa hết chúng.

Tính tay:

```
"1112"  -> '2' cuối ở chỉ số 3, không có gì sau nó      -> "1112"
"221"   -> '2' cuối ở chỉ số 1, sau nó có '1'            -> bỏ '1' -> "22"
"1"     -> không có '2'                                   -> ""
"2121"  -> '2' cuối ở chỉ số 2, sau nó có '1'            -> bỏ '1' -> "212"
```

### Bảng tính tay

| s | chỉ số '2' cuối | kết quả `s[:idx+1]` |
|---|---|---|
| `1112` | 3 | `1112` |
| `221` | 1 | `22` |
| `1` | -1 (không có) | `` (rỗng) |
| `2121` | 2 | `212` |
| `2111` | 0 | `2` |
| `1111` | -1 | `` |

In [6]:
for s in ["1112", "221", "1", "2121", "2111", "1111"]:
    idx = s.rfind('2')
    print(f"s={s!r:7} idx cuối của '2' = {idx:>2} -> {s[:idx + 1]!r}")

s='1112'  idx cuối của '2' =  3 -> '1112'
s='221'   idx cuối của '2' =  1 -> '22'
s='1'     idx cuối của '2' = -1 -> ''
s='2121'  idx cuối của '2' =  2 -> '212'
s='2111'  idx cuối của '2' =  0 -> '2'
s='1111'  idx cuối của '2' = -1 -> ''


### Vì sao "dài hơn" luôn thắng ở bài này?

Các chuỗi chỉ gồm `'1'` và `'2'` **không có số `0` ở đầu**, nên chuỗi có `L` chữ số luôn nằm trong khoảng `[10^(L-1), 10^L)`. Mọi số `L` chữ số đều lớn hơn mọi số `L - 1` chữ số. Vì thế ta ưu tiên **độ dài**, và chỉ khi hai ứng viên cùng độ dài mới cần so sánh từng chữ số.

In [7]:
import random
random.seed(11)
for _ in range(1000):
    a = "".join(random.choice("12") for _ in range(random.randint(1, 8)))
    b = "".join(random.choice("12") for _ in range(random.randint(1, 8)))
    if len(a) != len(b):
        assert (int(a) > int(b)) == (len(a) > len(b))
    assert (int(a) > int(b)) == ((len(a), a) > (len(b), b))
print("so sánh số dạng chuỗi bằng khóa (độ dài, chuỗi) là đúng")

so sánh số dạng chuỗi bằng khóa (độ dài, chuỗi) là đúng


## 5. Cách 1 — Brute Force / baseline

Thử **mọi dãy con** của `s` (mã hóa bằng mặt nạ bit từ `0` đến `2^n - 1`), giữ các dãy con **kết thúc bằng `'2'`**, rồi chọn dãy lớn nhất theo khóa `(độ dài, chuỗi)`. Đây là định nghĩa của đề dịch nguyên văn.

Nút thắt: `O(2^n * n)`. Chỉ khả thi cho `n <= 15` hay hơn; với `n = 100` là bất khả thi. Nhưng nó là **oracle** rất tốt để kiểm tra cách nhanh.

In [8]:
class SolutionBrute:
    def largestEven(self, s):
        n = len(s)
        best = ""
        for mask in range(1 << n):                      # mỗi mặt nạ là một cách chọn giữ/xóa
            sub = "".join(s[i] for i in range(n) if (mask >> i) & 1)
            if sub and sub[-1] == '2' and (len(sub), sub) > (len(best), best):
                best = sub
        return best

print(SolutionBrute().largestEven("1112"), SolutionBrute().largestEven("221"), repr(SolutionBrute().largestEven("1")))

1112 22 ''


**Độ phức tạp:** Time `O(2^n * n)`, Space `O(n)`. **Chỉ dùng cho `n` nhỏ.**

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần thử dãy con; đáp án là **tiền tố** của `s` kết thúc đúng tại `'2'` cuối cùng.

**Quy tắc "nếu… thì…":**

- Nếu `s` có chứa `'2'` -> đáp án là `s[0 .. idx]` với `idx` là vị trí `'2'` cuối cùng.
- Nếu `s` không chứa `'2'` -> đáp án `""`.

**Chứng minh ngắn:** mọi dãy con chẵn kết thúc bằng một `'2'` ở vị trí `p` nào đó trong `s`, nên độ dài của nó `<= p + 1`. Độ dài lớn nhất là `idx + 1` khi `p = idx` và giữ **mọi** ký tự trước đó, và đó là một dãy con hợp lệ (tiền tố). Mọi dãy con khác hoặc ngắn hơn (nên nhỏ hơn) hoặc bằng chính tiền tố này.

**Invariant (cách cắt dần):** sau mỗi lần cắt, chuỗi còn lại vẫn là tiền tố của `s` và chứa mọi `'2'` đã có từ đầu đến vị trí hiện tại.

In [9]:
# Kiểm chứng với vét cạn cho mọi chuỗi 1..12 ký tự
from itertools import product
cnt = 0
for n in range(1, 13):
    for t in product("12", repeat=n):
        s = "".join(t)
        if n <= 10:
            assert SolutionBrute().largestEven(s) == s[:s.rfind('2') + 1]
        cnt += 1
print("tiền tố tới '2' cuối khớp vét cạn; đã duyệt", cnt, "chuỗi")

tiền tố tới '2' cuối khớp vét cạn; đã duyệt 8190 chuỗi


## 7. Các cách tiếp theo

### Cách 2 — Cắt dần ký tự cuối khi nó không phải `'2'`

Lặp: khi chuỗi còn ký tự và ký tự cuối là `'1'` thì bỏ nó (`s = s[:-1]`). Dừng khi chuỗi rỗng hoặc ký tự cuối là `'2'`. Cách này bám sát suy luận "xóa các `'1'` ở cuối".

In [10]:
class SolutionTrim:
    def largestEven(self, s):
        while s and s[-1] != '2':
            s = s[:-1]
        return s

print(SolutionTrim().largestEven("1112"), SolutionTrim().largestEven("221"), repr(SolutionTrim().largestEven("1")))

1112 22 ''


**Độ phức tạp:** Time `O(n^2)` trong trường hợp xấu (mỗi lần cắt tạo chuỗi mới `O(n)`), Space `O(n)`. Dùng chỉ số thay vì cắt sẽ còn `O(n)`.

### Cách 3 — `rfind('2')` rồi lát cắt

`idx = s.rfind('2')` cho vị trí `'2'` cuối cùng (hoặc `-1`); đáp án `s[:idx + 1]`. Khi `idx = -1` thì `s[:0]` là chuỗi rỗng đúng ý đề, không cần `if`.

In [11]:
class SolutionRfind:
    def largestEven(self, s):
        idx = s.rfind('2')
        return s[:idx + 1]

print(SolutionRfind().largestEven("1112"), SolutionRfind().largestEven("221"), repr(SolutionRfind().largestEven("1")))

1112 22 ''


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho chuỗi kết quả.

### Cách 4 — `rstrip('1')`

Vì bảng chữ chỉ có `'1'` và `'2'`, bỏ mọi `'1'` ở cuối chính là dừng tại `'2'` cuối cùng. `s.rstrip('1')` làm đúng việc đó trong một dòng. **Chỉ đúng** nhờ ràng buộc "chỉ gồm `1` và `2`".

In [12]:
class SolutionRstrip:
    def largestEven(self, s):
        return s.rstrip('1')

print(SolutionRstrip().largestEven("1112"), SolutionRstrip().largestEven("221"), repr(SolutionRstrip().largestEven("1")))

1112 22 ''


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 2 (cắt dần) với `s = "2111"`:

| vòng | `s` trước | `s[-1]` | hành động | `s` sau |
|---|---|---|---|---|
| 1 | `2111` | `1` | cắt | `211` |
| 2 | `211` | `1` | cắt | `21` |
| 3 | `21` | `1` | cắt | `2` |
| dừng | `2` | `2` | giữ nguyên | `2` |

Trace `s = "1"`: `s[-1] = '1'` -> cắt thành `""`; vòng lặp dừng vì chuỗi rỗng, trả `""`.

In [13]:
def trace_trim(s):
    print(f"s = {s!r}")
    step = 0
    while s and s[-1] != '2':
        step += 1
        print(f"  vòng {step}: ký tự cuối {s[-1]!r} khác '2' -> cắt, còn {s[:-1]!r}")
        s = s[:-1]
    print(f"  dừng: s = {s!r}")
    return s

assert trace_trim("2111") == "2"
assert trace_trim("1") == ""
assert trace_trim("1212") == "1212"

s = '2111'
  vòng 1: ký tự cuối '1' khác '2' -> cắt, còn '211'
  vòng 2: ký tự cuối '1' khác '2' -> cắt, còn '21'
  vòng 3: ký tự cuối '1' khác '2' -> cắt, còn '2'
  dừng: s = '2'
s = '1'
  vòng 1: ký tự cuối '1' khác '2' -> cắt, còn ''
  dừng: s = ''
s = '1212'
  dừng: s = '1212'


In [14]:
def trace_rfind(s):
    idx = s.rfind('2')
    print(f"s={s!r}: rfind('2') = {idx} -> s[:{idx + 1}] = {s[:idx + 1]!r}")
    return s[:idx + 1]

assert trace_rfind("221") == "22"
assert trace_rfind("1111") == ""
assert trace_rfind("2121") == "212"

s='221': rfind('2') = 1 -> s[:2] = '22'
s='1111': rfind('2') = -1 -> s[:0] = ''
s='2121': rfind('2') = 2 -> s[:3] = '212'


## 9. Code nộp LeetCode

In [15]:
from typing import List, Optional

class Solution:
    def largestEven(self, s: str) -> str:
        idx = s.rfind('2')           # vị trí '2' cuối cùng; -1 nếu không có
        return s[:idx + 1]           # giữ mọi ký tự đến hết '2' cuối cùng

## 10. Vì sao thuật toán đúng

- **Hợp lệ:** `s[:idx + 1]` là tiền tố của `s`, nên là dãy con hợp lệ (xóa các ký tự sau `idx`). Nó kết thúc bằng `'2'` nên là số chẵn. Nếu không có `'2'` thì `idx = -1`, kết quả `""`, đúng theo đề.
- **Tối ưu về độ dài:** gọi một đáp án bất kỳ là dãy con chẵn `t` kết thúc bằng `'2'` ở vị trí `p` của `s`. Mọi ký tự của `t` đều nằm ở chỉ số `<= p`, nên `|t| <= p + 1 <= idx + 1`. Vậy `s[:idx + 1]` có độ dài lớn nhất trong các dãy con chẵn.
- **Tối ưu về giá trị:** các chuỗi chỉ gồm `1`, `2` không có `0` đầu, nên chuỗi dài hơn là số lớn hơn. Nếu một dãy con chẵn khác `t` cũng có độ dài `idx + 1` thì phải dùng mọi ký tự `s[0..idx]`, tức `t = s[:idx + 1]`. Do đó đáp án là **duy nhất** lớn nhất.
- **Các cách khác:** cách cắt dần và `rstrip('1')` cùng dừng tại `'2'` cuối cùng nên cho cùng kết quả; cách vét cạn kiểm tra trực tiếp định nghĩa với `n` nhỏ (mục 6 và 12).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Vét cạn dãy con | `O(2^n * n)` | `O(n)` |
| Cắt dần ký tự cuối | `O(n^2)` (tạo chuỗi mới) hoặc `O(n)` | `O(n)` |
| `rfind` + lát cắt | `O(n)` | `O(n)` |
| `rstrip('1')` | `O(n)` | `O(n)` |

`n = len(s) <= 100`. Chỉ cách vét cạn là không khả thi với `n` lớn; ba cách còn lại là tuyến tính.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, toàn `'1'`, toàn `'2'`, `'2'` ở đầu và `'1'` ở cuối) và so sánh **tất cả cách**; thêm oracle vét cạn cho chuỗi ngắn và kiểm tra tính chất: kết quả là tiền tố, chẵn hoặc rỗng.

In [16]:
impls = [SolutionTrim(), SolutionRfind(), SolutionRstrip(), Solution()]

cases = [
    ("1112", "1112"), ("221", "22"), ("1", ""), ("2", "2"), ("12", "12"), ("21", "2"),
    ("1111", ""), ("2222", "2222"), ("2111", "2"), ("2121", "212"), ("1" * 99 + "2", "1" * 99 + "2"),
    ("2" + "1" * 99, "2"), ("1" * 100, ""), ("12" * 50, "12" * 50), ("21" * 50, "21" * 49 + "2"),
]
for s, exp in cases:
    assert 1 <= len(s) <= 100 and set(s) <= {'1', '2'}
    for impl in impls:
        got = impl.largestEven(s)
        assert got == exp, (type(impl).__name__, s, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 15 ca


In [17]:
import random
from itertools import product

# Oracle vét cạn cho mọi chuỗi độ dài 1..10
for n in range(1, 11):
    for t in product("12", repeat=n):
        s = "".join(t)
        exp = SolutionBrute().largestEven(s)
        assert all(impl.largestEven(s) == exp for impl in impls), s
print("vét cạn mọi chuỗi độ dài 1..10 OK (4 cách nhanh khớp oracle)")

random.seed(3798)
for _ in range(1500):
    n = random.randint(1, 100)
    p = random.choice([0.1, 0.5, 0.9])
    s = "".join('2' if random.random() < p else '1' for _ in range(n))
    outs = [impl.largestEven(s) for impl in impls]
    assert all(o == outs[0] for o in outs), (s, outs)
    r = outs[0]
    assert s.startswith(r)                                  # là tiền tố -> là dãy con
    assert r == "" or (r[-1] == '2' and int(r) % 2 == 0)    # chẵn hoặc rỗng
    assert ('2' in s) == (r != "")
    assert r == "" or '2' not in s[len(r):]                 # không còn '2' phía sau (đã lấy hết)
print("1500 test ngẫu nhiên (seed=3798) OK: giống nhau, là tiền tố, chẵn hoặc rỗng")

vét cạn mọi chuỗi độ dài 1..10 OK (4 cách nhanh khớp oracle)
1500 test ngẫu nhiên (seed=3798) OK: giống nhau, là tiền tố, chẵn hoặc rỗng


In [18]:
# Thêm: các chuỗi đặc biệt cho thấy đáp án chỉ phụ thuộc vị trí '2' cuối cùng
for n in [1, 2, 7, 50, 100]:
    assert Solution().largestEven("1" * n) == ""
    assert Solution().largestEven("2" * n) == "2" * n
    assert Solution().largestEven("2" + "1" * (n - 1)) == "2"
    assert Solution().largestEven("1" * (n - 1) + "2") == "1" * (n - 1) + "2"
print("các mẫu toàn '1', toàn '2', '2' đầu, '2' cuối OK")

các mẫu toàn '1', toàn '2', '2' đầu, '2' cuối OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: xóa mọi ký tự `'1'`

Xóa `'1'` làm số **ngắn đi** nên nhỏ hơn: `"1112"` cần giữ nguyên chứ không phải `"2"`.

In [19]:
def wrong_remove_all_ones(s):
    return s.replace('1', '') if '2' in s else ""        # SAI: bỏ cả các '1' ở phía trước

print(wrong_remove_all_ones("1112"), "!=", Solution().largestEven("1112"))
assert wrong_remove_all_ones("1112") == "2" and Solution().largestEven("1112") == "1112"

2 != 1112


### Lỗi 2: cắt đến lần xuất hiện **đầu tiên** của `'2'`

`find` trả vị trí đầu tiên; nhưng ta cần vị trí **cuối cùng** để giữ được nhiều ký tự nhất.

In [20]:
def wrong_first_two(s):
    return s[:s.find('2') + 1]                           # SAI: find thay vì rfind

print(wrong_first_two("221"), "!=", Solution().largestEven("221"))
assert wrong_first_two("221") == "2" and Solution().largestEven("221") == "22" 

2 != 22


### Lỗi 3: kiểm tra cả số chẵn lẻ rồi bỏ cuộc

Nếu `int(s)` lẻ thì chưa hết cách: còn có thể **xóa** ký tự cuối để được số chẵn.

In [21]:
def wrong_check_whole(s):
    return s if int(s) % 2 == 0 else ""                  # SAI: không thử xóa bớt

print(repr(wrong_check_whole("221")), "!=", repr(Solution().largestEven("221")))
assert wrong_check_whole("221") == "" and Solution().largestEven("221") == "22" 

'' != '22'


### Lỗi 4: chỉ xóa đúng một ký tự cuối

Có thể còn nhiều `'1'` liên tiếp ở cuối cần xóa hết.

In [22]:
def wrong_one_removal(s):
    if s and s[-1] == '1':
        s = s[:-1]                                       # SAI: chỉ xóa một lần
    return s if s.endswith('2') else ""                  # sai khi còn nhiều '1' ở cuối: "2111" -> "211" không chẵn nhưng bị xử lý lệch

print(repr(wrong_one_removal("2111")), "!=", repr(Solution().largestEven("2111")))      # "" vs "2"
assert wrong_one_removal("2111") == "" and Solution().largestEven("2111") == "2" 

'' != '2'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số lớn nhất tạo từ việc xóa ký tự" + điều kiện về **chữ số cuối** -> chỉ xử lý **phần đuôi**; phần đầu giữ nguyên để số dài nhất.
- Số dài hơn (không `0` đầu) luôn lớn hơn: ưu tiên **độ dài** trước, rồi mới so từ trái sang phải.
- Muốn "dãy con dài nhất kết thúc bằng X" -> tìm vị trí **cuối cùng** của X (`rfind`), lấy tiền tố.
- `s[:idx + 1]` với `idx = -1` cho chuỗi rỗng, tránh được `if`.
- `rstrip` chỉ dùng khi bảng chữ hạn chế (ở đây chỉ `1` và `2`); nếu có thêm chữ số chẵn khác (`4, 6, 8, 0`) phải dùng vòng lặp kiểm tra tập chữ số chẵn.
- Tự kiểm tra: kết quả phải là **tiền tố** của `s` và kết thúc bằng `'2'` (hoặc rỗng).

## 15. Biến thể của bài

1. **Largest Odd Number in String** (LeetCode 1903): chuỗi số bất kỳ, tìm **chuỗi con** lẻ lớn nhất (tiền tố đến chữ số lẻ cuối cùng).
2. **Remove Trailing Zeros From a String** (2710): bỏ các `'0'` ở cuối.
3. **Maximum Odd Binary Number** (2864): sắp xếp lại các bit để được số nhị phân lẻ lớn nhất.
4. **Remove Digit From Number to Maximize Result** (2259): xóa đúng một chữ số `d` để số lớn nhất.
5. **Largest Even Number** với bảng chữ rộng hơn (`0..9`): tiền tố đến chữ số chẵn cuối cùng, nhưng phải cẩn thận số `0` đứng đầu.
6. Biến thể "số lẻ lớn nhất": chỉ đổi `'2'` thành `'1'` trong kiểm tra.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Vét cạn dãy con | thử mọi tập ký tự giữ lại | `O(2^n * n)` | `O(n)` | Oracle kiểm tra, `n <= 15` |
| Cắt dần cuối | bỏ `'1'` ở cuối đến khi gặp `'2'` | `O(n)`-`O(n^2)` | `O(n)` | Giải thích trực giác |
| `rfind` + lát cắt | `s[:rfind('2') + 1]` | `O(n)` | `O(n)` | Nộp bài, tổng quát hơn `rstrip` |
| `rstrip('1')` | bỏ mọi `'1'` cuối | `O(n)` | `O(n)` | Một dòng, chỉ khi bảng chữ là `{1, 2}` |

Nộp cách `rfind`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().largestEven("12121")` ra gì?

**Bài 2 (tính tay):** với `s = "1121211"`, tìm chỉ số `'2'` cuối cùng và đáp án.

**Bài 3 (điền TODO):** hoàn thành `largest_odd(s)` cho bài 1903 dạng đơn giản: `s` chỉ gồm `'1'` và `'2'`, trả về dãy con **lẻ** lớn nhất (tiền tố kết thúc ở `'1'` cuối cùng).

**Bài 4 (tự viết lại):** viết `largest_even_digits(s)` cho chuỗi gồm chữ số `0..9` **không có `0` ở đầu**: trả về tiền tố đến chữ số chẵn cuối cùng (hoặc `""`).

**Bài 5 (mini challenge):** viết `count_even_prefixes(s)` đếm số tiền tố của `s` biểu diễn số chẵn (kết thúc bằng `'2'`).

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(repr(Solution().largestEven("12121")))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def largest_odd_todo(s):
    # TODO: dùng rfind('1') và lát cắt, giống largestEven nhưng cho số lẻ
    return ""

'1212'


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: ký tự cuối là `'1'`; hãy xem `'2'` cuối cùng nằm ở đâu. Bài 2: đếm chỉ số từ `0`.

**Mức 2 (vừa):** Bài 3: số lẻ kết thúc bằng `'1'`, nên dùng `rfind('1')`. Bài 4: duyệt từ phải sang trái tìm chữ số chẵn đầu tiên. Bài 5: tiền tố độ dài `k` kết thúc ở ký tự `s[k - 1]`.

**Mức 3 (gần đáp án):** Bài 1: `'2'` cuối ở chỉ số 3 -> `"1212"`. Bài 2: `s = 1121211`, `'2'` cuối ở chỉ số 4 -> `"11212"`. Bài 3: `s[:s.rfind('1') + 1]`. Bài 5: `sum(1 for ch in s if ch == '2')`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().largestEven("12121") == "1212"
assert "1121211".rfind('2') == 4 and Solution().largestEven("1121211") == "11212"

# Bài 3
def largest_odd(s):
    return s[:s.rfind('1') + 1]
assert largest_odd("221") == "221" and largest_odd("222") == "" and largest_odd("1222") == "1"

# Bài 4
def largest_even_digits(s):
    for i in range(len(s) - 1, -1, -1):
        if s[i] in "02468":
            return s[:i + 1]
    return ""
assert largest_even_digits("1357") == "" and largest_even_digits("1352") == "1352" and largest_even_digits("2357") == "2"
assert all(largest_even_digits(s) == Solution().largestEven(s) for s in ["1112", "221", "1", "2121"])

# Bài 5
def count_even_prefixes(s):
    return sum(1 for ch in s if ch == '2')
assert count_even_prefixes("1212") == 2 and count_even_prefixes("111") == 0
assert all(count_even_prefixes(s) == sum(1 for k in range(1, len(s) + 1) if s[:k][-1] == '2') for s in ["1212", "2221", "1"])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "dãy con dài nhất kết thúc bằng ký tự thỏa điều kiện":**

```python
idx = s.rfind(target)        # vị trí cuối cùng của ký tự thỏa điều kiện
return s[:idx + 1]           # idx = -1 -> chuỗi rỗng, không cần if
# nhiều ký tự thỏa điều kiện -> duyệt từ phải sang trái tìm cái đầu tiên thỏa
```

**Checklist trước khi submit:**

- [ ] Hiểu "xóa ký tự giữ nguyên thứ tự" = chọn dãy con, và lớn nhất = **dài nhất** (không `0` đầu)?
- [ ] Dùng `rfind` (cuối cùng), không phải `find` (đầu tiên)?
- [ ] Trường hợp không có `'2'` trả `""` (không `None`, không lỗi)?
- [ ] Thử `"1"`, `"2"`, `"1111"`, `"2111"`, chuỗi dài 100?
- [ ] Đã so sánh với oracle vét cạn trên mọi chuỗi ngắn?
- [ ] Chữ ký đúng `largestEven(self, s: str) -> str`?